<a href="https://colab.research.google.com/github/naomiv2910-cell/curso-agentes-ia/blob/main/unidad-01/primer_agente_smolagents_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Preparación del entorno

Para comenzar, instalé las librerías necesarias para desarrollar el agente. Utilicé smolagents, que es la librería de Hugging Face con la que voy a trabajar, y google-genai para conectarme con Gemini.

Durante la instalación había encontrado algunas incompatibilidades entre las versiones de las dependencias que trae Google Colab y las que necesitaba el proyecto. Por eso ajusté las versiones de google-auth y google-genai para poder continuar sin esos conflictos.

In [2]:
%pip install -q "google-auth==2.49.0" "google-genai>=2.12.1,<3" smolagents

### Verificación de las librerías

Después de la instalación, importé las librerías que iba a utilizar. Principalmente CodeAgen`, para construir el agente, y tool, que permite convertir funciones de Python en herramientas que el agente puede utilizar.

La importación se realizó correctamente, así que pude continuar con la configuración del modelo.

In [3]:
from google import genai
from smolagents import CodeAgent, tool

print("¡Librerías importadas correctamente!")


¡Librerías importadas correctamente!


### Conexión con Gemini

Para esta práctica decidí utilizar Gemini como modelo de lenguaje, aprovechando que ya había trabajado con su API en el notebook anterior del curso.

Guardé la API key en los secretos de Google Colab para no tener que escribirla directamente en el código. Después creé el cliente de Gemini y comprobé que la configuración se realizara correctamente.

De esta manera puedo trabajar con el modelo sin dejar expuesta mi clave cuando suba el notebook a GitHub.

In [4]:

from google.colab import userdata
from google import genai

# Recuperar la API key guardada en Colab
API_KEY = userdata.get("GEMINI_API_KEY")

# Crear el cliente de Gemini
client = genai.Client(api_key=API_KEY)

print("Gemini configurado correctamente")


Gemini configurado correctamente


### Configuración inicial del modelo

En esta parte utilicé LiteLLMModel para conectar Gemini con smolagents. Inicialmente había intentado trabajar con Gemini 2.5 Flash, pero recibí un error 404 porque el modelo ya no estaba disponible para nuevos usuarios.

El propio mensaje de la API recomendaba utilizar Gemini 3.8 Flash, así que actualicé la configuración.

Aunque el modelo quedó configurado en Python, todavía faltaba comprobar que pudiera responder correctamente cuando ejecutara el agente.

In [11]:
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id="gemini/gemini-3.8-flash",
    api_key=API_KEY
)

print("Modelo configurado para smolagents")

Modelo configurado para smolagents


### Creación de mi primera herramienta

Para probar el funcionamiento del agente quise utilizar un ejemplo relacionado con mi emprendimiento de trenzas.

Creé una función llamada "calcular_precio_trenzas", que recibe la cantidad de trenzas y el precio de cada una, y devuelve el valor total del servicio.

Lo interesante de esta parte fue utilizar @tool, porque permite que una función normal de Python se convierta en una herramienta que el agente puede reconocer y ejecutar.

Con esto empecé a entender mejor la diferencia entre pedirle a un modelo que responda una pregunta y darle la posibilidad de utilizar herramientas para resolverla.

In [8]:
from smolagents import tool

@tool
def calcular_precio_trenzas(
    cantidad: int,
    precio_por_trenza: float
) -> float:
    """
    Calcula el precio total de un servicio de trenzado.

    Args:
        cantidad: Número de trenzas solicitadas.
        precio_por_trenza: Precio de cada trenza en pesos colombianos.
    """
    return cantidad * precio_por_trenza

print("Herramienta creada correctamente")

Herramienta creada correctamente


### Construcción del agente

Una vez creada la herramienta, construí el agente utilizando CodeAgent.

Le asigné el modelo de Gemini y la función para calcular precios. También establecí un máximo de tres pasos de ejecución, para que el agente tuviera un límite al intentar resolver la tarea.

En este punto ya tenía definidos los componentes principales: el modelo de lenguaje, la herramienta y el agente que se encargaría de utilizarlos.

In [12]:
from smolagents import CodeAgent

agent = CodeAgent(
    tools=[calcular_precio_trenzas],
    model=model,
    max_steps=3
)

print("Agente actualizado correctamente")


Agente actualizado correctamente


### Primera prueba del agente

En la primera prueba le pedí al agente que calculara el precio de ocho trenzas, cada una con un valor de $6.000.

Sin embargo, la ejecución tardó bastante y terminó mostrando un error 503 Service Unavailable. Al revisar el mensaje, encontré que Gemini 3.8 Flash estaba experimentando una demanda muy alta y no podía procesar la solicitud en ese momento.

Este problema no estaba relacionado directamente con la función que había creado, sino con la disponibilidad del modelo a través de la API.

En lugar de seguir ejecutando la misma solicitud, decidí buscar otra versión de Gemini que pudiera utilizar para terminar el ejercicio.

In [13]:

resultado = agent.run(
    "Una clienta quiere hacerse 8 trenzas. "
    "Cada trenza cuesta 6000 pesos colombianos. "
    "Utiliza tu herramienta para calcular el precio total."
)

print("Respuesta final:", resultado)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Una clienta quiere hacerse 8 trenzas. Cada trenza cuesta 6000 pesos colombianos. Utiliza tu herramienta para    │
│ calcular el precio total.                                                                                       │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.8-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm.turn_on_debug()'.



Error in generating model output:
litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please 
try again later.",
    "status": "UNAVAILABLE"
  }
}

[Step 1: Duration 345.90 seconds]

AgentGenerationError: Error in generating model output:
litellm.ServiceUnavailableError: GeminiException - {
  "error": {
    "code": 503,
    "message": "This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.",
    "status": "UNAVAILABLE"
  }
}


### Búsqueda de otro modelo

Para solucionar el problema anterior, consulté los modelos disponibles mediante la API de Gemini.

Filtré los resultados para mostrar las versiones Flash, ya que buscaba un modelo que pudiera utilizar en una tarea relativamente sencilla sin necesitar demasiados recursos.

La consulta devolvió varias alternativas, entre ellas Gemini 3.1 Flash Lite, que decidí probar porque era una versión más ligera y ya la había utilizado en el primer notebook del curso.

In [14]:

from google import genai

client = genai.Client(api_key=API_KEY)

modelos = [
    m.name
    for m in client.models.list()
    if "generateContent" in (m.supported_actions or [])
]

for nombre in modelos:
    if "flash" in nombre.lower():
        print(nombre)


models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-omni-1.2-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts


### Comprobación del nuevo modelo

Antes de volver a ejecutar todo el agente, hice una prueba sencilla para comprobar que Gemini 3.1 Flash Lite respondiera correctamente.

Le pedí que respondiera únicamente con la palabra "Hola", y esta vez la solicitud funcionó.

Esta prueba me permitió confirmar que podía comunicarme con el modelo mediante la API. A partir de ahí pude continuar con la integración en `smolagents`.

In [15]:

from google import genai

client = genai.Client(api_key=API_KEY)

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Responde únicamente: Hola"
)

print(response.text)


Hola


### Actualización de la configuración

Después de comprobar que Gemini 3.1 Flash Lite funcionaba, actualicé el modelo utilizado por el agente.

También configuré un tiempo máximo de espera de 30 segundos y desactivé los reintentos automáticos del adaptador, para evitar que una solicitud volviera a quedarse ejecutándose durante varios minutos.

No tuve que modificar la herramienta que había creado anteriormente. Simplemente cambié el modelo y volví a construir el agente con la nueva configuración.

In [16]:

from smolagents import LiteLLMModel, CodeAgent

model = LiteLLMModel(
    model_id="gemini/gemini-3.1-flash-lite",
    api_key=API_KEY,
    timeout=30,
    num_retries=0
)

agent = CodeAgent(
    tools=[calcular_precio_trenzas],
    model=model,
    max_steps=3
)

print("Agente actualizado correctamente")


Agente actualizado correctamente


### Resultado final

Finalmente, volví a ejecutar el agente con la solicitud de calcular el precio de ocho trenzas a $6.000 cada una.

Esta vez el agente consiguió completar la tarea, aunque durante los primeros dos pasos presentó errores al interpretar el código generado. En el tercer intento logró ejecutar correctamente la función calcular_precio_trenzas y obtuvo una respuesta final de **$48.000**.

Me pareció interesante que el agente no resolviera todo perfectamente desde el primer intento, porque pude observar cómo utiliza los pasos disponibles para intentar completar una tarea.

También pude comprobar que realmente utilizó la herramienta que había definido y no se limitó a generar una respuesta en texto.

El resultado final coincidió con el cálculo esperado, así que considero que logré el objetivo principal de esta práctica.

In [17]:

resultado = agent.run(
    "Una clienta quiere 8 trenzas a 6000 pesos cada una. "
    "Usa la herramienta calcular_precio_trenzas "
    "para obtener el precio total."
)

print("Respuesta final:", resultado)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Una clienta quiere 8 trenzas a 6000 pesos cada una. Usa la herramienta calcular_precio_trenzas para obtener el  │
│ precio total.                                                                                                   │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-3.1-flash-lite ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
expected string or bytes-like object, got 'NoneType'
Make sure to provide correct code blobs.

[Step 1: Duration 15.26 seconds| Input tokens: 2,257 | Output tokens: 27]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
expected string or bytes-like object, got 'NoneType'
Make sure to provide correct code blobs.

... (2 outputs left out)

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  total_price = calcular_precio_trenzas(cantidad=8, precio_por_trenza=6000)                                        
  final_answer(total_price)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 48000

[Step 3: Duration 1.99 seconds| Input tokens: 6,956 | Output tokens: 159]

Respuesta final: 48000


## Conclusiones

Con este ejercicio pude construir mi primer agente utilizando `smolagents` y Gemini. Aunque al principio parecía una tarea sencilla, durante el desarrollo encontré varios problemas relacionados con las dependencias, la disponibilidad de los modelos y la generación del código.

Lo que más me ayudó a entender el funcionamiento de los agentes fue ver cómo se conectan el modelo de lenguaje y las herramientas. Antes había trabajado con modelos que recibían una instrucción y generaban una respuesta, pero en este caso pude observar cómo el agente utiliza una función de Python para realizar una operación.

También entendí que no siempre basta con tener el código bien escrito. Al trabajar con modelos externos pueden aparecer problemas de disponibilidad, compatibilidad o límites de uso, y es importante saber identificar de dónde viene cada error.

En general, esta práctica me permitió relacionar los conceptos de la Unidad 1 de Hugging Face con un ejemplo de mi vida cotidiana. Me gustó poder utilizar una situación de mi emprendimiento para comprender mejor cómo funciona un agente y cómo podría aplicar este tipo de herramientas en proyectos más grandes.